# Cross-Dataset Generalization: B-Point Regression

Addresses the reviewer remark that all cross-validation folds in `B_Point_Regression.ipynb` are drawn from the combined (EmpkinS + Guardian) dataset, so no fold ever tests generalization across studies. Here the model is instead trained on one study and evaluated on the other, in both directions:

1. Train on EmpkinS (`VP_*`), test on Guardian (`GDN*`)
2. Train on Guardian (`GDN*`), test on EmpkinS (`VP_*`)

For context, a within-dataset baseline (5-fold group CV, restricted to each single dataset) is also computed with the identical fixed hyperparameters, so the cross-dataset generalization gap can be judged against the within-dataset performance ceiling of the same model.

In [3]:
# Imports
import warnings
from pathlib import Path

import biopsykit as bp
import numpy as np
import pandas as pd
import pingouin as pg
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model._cd_fast import ConvergenceWarning
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [4]:
# Datapaths
data_path = Path("../../results")
rater = "rater_01"

In [5]:
# Load data (combined EmpkinS + Guardian dataset, same input as B_Point_Regression.ipynb)
input_data_b_point = pd.read_csv(
    data_path.joinpath(f"rr-interval/{rater}/train_data_b_point_rr_interval_include_nan_three_best_only.csv"),
    index_col=[0, 1, 2, 3, 4, 5],
).astype(float)
input_data_b_point.head()

,,,,,,rr_interval_ms_estimated,debski1993-second-derivative,drost2022,lozano2007-linear-regression
,participant,condition,phase,heartbeat_id_reference,b_point_sample_reference,,,,
0,GDN0005,Dummy,HoldingBreath,0,388.0,850.0,452.0,400.0,412.0
1,GDN0005,Dummy,HoldingBreath,1,404.0,778.0,388.0,420.0,404.0
2,GDN0005,Dummy,HoldingBreath,3,376.0,746.0,366.0,386.0,366.0
3,GDN0005,Dummy,HoldingBreath,4,390.0,766.0,376.0,396.0,372.0
4,GDN0005,Dummy,HoldingBreath,5,386.0,790.0,418.0,392.0,378.0


## Split by study

EmpkinS participants are labeled `VP_*`, Guardian participants are labeled `GDN*` (cf. `preprocessing/Preprocessing_B_Point.ipynb`).

In [6]:
# Assign each row to its originating study based on the participant ID prefix
participant_ids = input_data_b_point.index.get_level_values("participant")
is_empkins = participant_ids.str.startswith("VP")
is_guardian = participant_ids.str.startswith("GDN")
assert (is_empkins | is_guardian).all(), "Found participant IDs that belong to neither study"

data_empkins = input_data_b_point.loc[is_empkins]
data_guardian = input_data_b_point.loc[is_guardian]

print(
    f"EmpkinS: {data_empkins.shape[0]} heartbeats, "
    f"{data_empkins.index.get_level_values('participant').nunique()} participants"
)
print(
    f"Guardian: {data_guardian.shape[0]} heartbeats, "
    f"{data_guardian.index.get_level_values('participant').nunique()} participants"
)

EmpkinS: 4995 heartbeats, 15 participants
Guardian: 6244 heartbeats, 24 participants


## Fixed model configuration

Same fixed, best-performing hyperparameter configuration as in `B_Point_Regression.ipynb` (cf. `Computational_Cost.ipynb` / `Analyse_Model_Parameters.ipynb`) — no hyperparameter search is performed here, only the train/test assignment changes.

In [7]:
def make_pipeline() -> Pipeline:
    """Create the MinMaxScaler + RandomForestRegressor pipeline used for the cross-dataset evaluation."""
    return Pipeline(
        [
            ("scaler", MinMaxScaler()),
            (
                "clf",
                RandomForestRegressor(
                    bootstrap=True,
                    criterion="friedman_mse",
                    max_features=0.8,
                    n_estimators=250,
                    ccp_alpha=0.001,
                    random_state=0,
                ),
            ),
        ]
    )

## Cross-dataset evaluation helper

Trains the fixed pipeline on `train_df` and evaluates it on the entirely held-out `test_df` (a different study), reporting sample-wise and subject-wise MAE.

In [8]:
from collections.abc import Callable


def evaluate_cross_dataset(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    train_name: str,
    test_name: str,
    pipeline_factory: Callable[[], Pipeline] = make_pipeline,
) -> dict[str, object]:
    """Train a pipeline on one dataset, evaluate it on the other and report sample- and subject-wise MAE."""
    x_train, y_train, _, _ = bp.classification.utils.prepare_df_sklearn(
        data=train_df, label_col="b_point_sample_reference", subject_col="participant", print_summary=False
    )
    x_test, y_test, _, _ = bp.classification.utils.prepare_df_sklearn(
        data=test_df, label_col="b_point_sample_reference", subject_col="participant", print_summary=False
    )

    pipeline = pipeline_factory()
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)

    abs_error = np.abs(y_test - y_pred)
    mae_sample_wise = np.mean(abs_error)
    std_sample_wise = np.std(abs_error)
    print(f"Train: {train_name} (n={len(y_train)}) -> Test: {test_name} (n={len(y_test)})")
    print(f"Sample-wise MAE: {mae_sample_wise:.2f} ± {std_sample_wise:.2f} ms")

    abs_error_series = pd.Series(abs_error, index=test_df.index, name="abs_error")
    subject_mae = abs_error_series.groupby("participant").mean()
    print(f"Subject-wise MAE: {subject_mae.mean():.2f} ± {subject_mae.std():.2f} ms")

    return {
        "train_name": train_name,
        "test_name": test_name,
        "pipeline": pipeline,
        "X_train": x_train,
        "y_train": y_train,
        "X_test": x_test,
        "y_test": y_test,
        "y_pred": y_pred,
        "abs_error": abs_error_series,
        "subject_mae": subject_mae,
        "mae_sample_wise": mae_sample_wise,
        "std_sample_wise": std_sample_wise,
    }

## Direction 1: Train on EmpkinS &rarr; Test on Guardian

In [9]:
results_empkins_to_guardian = evaluate_cross_dataset(
    train_df=data_empkins, test_df=data_guardian, train_name="EmpkinS", test_name="Guardian"
)

Train: EmpkinS (n=4995) -> Test: Guardian (n=6244)
Sample-wise MAE: 26.67 ± 34.70 ms
Subject-wise MAE: 29.77 ± 22.25 ms


## Direction 2: Train on Guardian &rarr; Test on EmpkinS

In [10]:
results_guardian_to_empkins = evaluate_cross_dataset(
    train_df=data_guardian, test_df=data_empkins, train_name="Guardian", test_name="EmpkinS"
)

Train: Guardian (n=6244) -> Test: EmpkinS (n=4995)
Sample-wise MAE: 60.13 ± 56.86 ms
Subject-wise MAE: 52.92 ± 39.35 ms


## Within-dataset baseline (same fixed hyperparameters)

For reference, 5-fold group CV performance of the identical pipeline when trained and evaluated within a single study, so the cross-dataset MAE above can be compared against the within-dataset ceiling instead of the combined-dataset numbers reported in `B_Point_Regression.ipynb`.

In [11]:
def evaluate_within_dataset(df: pd.DataFrame, name: str, n_splits: int = 5) -> tuple[pd.Series, pd.Series]:
    """Evaluate the pipeline within one dataset using grouped k-fold cross-validation."""
    x, y, groups, _ = bp.classification.utils.prepare_df_sklearn(
        data=df, label_col="b_point_sample_reference", subject_col="participant", print_summary=False
    )
    pipeline = make_pipeline()
    cv = GroupKFold(n_splits=n_splits)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        y_pred = cross_val_predict(pipeline, x, y, cv=cv, groups=groups)

    abs_error = np.abs(y - y_pred)
    print(f"{name} within-dataset {n_splits}-fold CV (n={len(y)})")
    print(f"Sample-wise MAE: {np.mean(abs_error):.2f} \u00b1 {np.std(abs_error):.2f} ms")

    abs_error_series = pd.Series(abs_error, index=df.index, name="abs_error")
    subject_mae = abs_error_series.groupby("participant").mean()
    print(f"Subject-wise MAE: {subject_mae.mean():.2f} \u00b1 {subject_mae.std():.2f} ms")
    return abs_error_series, subject_mae


baseline_empkins_abs_error, baseline_empkins_subject_mae = evaluate_within_dataset(data_empkins, "EmpkinS")
print()
baseline_guardian_abs_error, baseline_guardian_subject_mae = evaluate_within_dataset(data_guardian, "Guardian")

EmpkinS within-dataset 5-fold CV (n=4995)
Sample-wise MAE: 9.05 ± 11.21 ms
Subject-wise MAE: 9.31 ± 3.32 ms

Guardian within-dataset 5-fold CV (n=6244)
Sample-wise MAE: 11.05 ± 13.71 ms
Subject-wise MAE: 11.58 ± 3.89 ms


## Summary

Sample-wise and subject-wise MAE for both cross-dataset generalization directions, alongside the matching within-dataset baseline.

In [12]:
def fmt_mean_std(mean: float, std: float) -> str:
    """Format a mean and standard deviation as ``mean ± std``."""
    return f"{mean:.2f} ± {std:.2f}"


summary_df = pd.DataFrame(
    [
        {
            "setting": "EmpkinS -> Guardian (cross-dataset)",
            "sample_wise_mae": fmt_mean_std(
                results_empkins_to_guardian["abs_error"].mean(), results_empkins_to_guardian["abs_error"].std()
            ),
            "subject_wise_mae": fmt_mean_std(
                results_empkins_to_guardian["subject_mae"].mean(), results_empkins_to_guardian["subject_mae"].std()
            ),
        },
        {
            "setting": "Guardian (within-dataset baseline)",
            "sample_wise_mae": fmt_mean_std(baseline_guardian_abs_error.mean(), baseline_guardian_abs_error.std()),
            "subject_wise_mae": fmt_mean_std(baseline_guardian_subject_mae.mean(), baseline_guardian_subject_mae.std()),
        },
        {
            "setting": "Guardian -> EmpkinS (cross-dataset)",
            "sample_wise_mae": fmt_mean_std(
                results_guardian_to_empkins["abs_error"].mean(), results_guardian_to_empkins["abs_error"].std()
            ),
            "subject_wise_mae": fmt_mean_std(
                results_guardian_to_empkins["subject_mae"].mean(), results_guardian_to_empkins["subject_mae"].std()
            ),
        },
        {
            "setting": "EmpkinS (within-dataset baseline)",
            "sample_wise_mae": fmt_mean_std(baseline_empkins_abs_error.mean(), baseline_empkins_abs_error.std()),
            "subject_wise_mae": fmt_mean_std(baseline_empkins_subject_mae.mean(), baseline_empkins_subject_mae.std()),
        },
    ]
)
summary_df

,setting,sample_wise_mae,subject_wise_mae
0,EmpkinS -> Guardian (cross-dataset),26.67 ± 34.71,29.77 ± 22.25
1,Guardian (within-dataset baseline),11.05 ± 13.71,11.58 ± 3.89
2,Guardian -> EmpkinS (cross-dataset),60.13 ± 56.86,52.92 ± 39.35
3,EmpkinS (within-dataset baseline),9.05 ± 11.21,9.31 ± 3.32


In [13]:
print(summary_df.to_latex(index=False).replace("±", r"$\pm$"))

\begin{tabular}{lll}
\toprule
setting & sample_wise_mae & subject_wise_mae \\
\midrule
EmpkinS -> Guardian (cross-dataset) & 26.67 $\pm$ 34.71 & 29.77 $\pm$ 22.25 \\
Guardian (within-dataset baseline) & 11.05 $\pm$ 13.71 & 11.58 $\pm$ 3.89 \\
Guardian -> EmpkinS (cross-dataset) & 60.13 $\pm$ 56.86 & 52.92 $\pm$ 39.35 \\
EmpkinS (within-dataset baseline) & 9.05 $\pm$ 11.21 & 9.31 $\pm$ 3.32 \\
\bottomrule
\end{tabular}



## Statistical comparison: cross-dataset vs. within-dataset error

Paired, subject-wise comparison (Wilcoxon signed-rank, cf. `Compute_Statistical_Significance.ipynb`) between the cross-dataset and within-dataset absolute errors, restricted to participants present in both settings for a given test study.

In [14]:
def compare_cross_vs_within(
    cross_subject_mae: pd.Series, within_subject_mae: pd.Series, test_name: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare subject-wise cross-dataset and within-dataset MAE with a Wilcoxon signed-rank test."""
    comparison = pd.concat(
        [cross_subject_mae.rename("cross_dataset_mae"), within_subject_mae.rename("within_dataset_mae")],
        axis=1,
        join="inner",
    )
    print(f"Test study: {test_name} ({comparison.shape[0]} matched participants)")
    normality_results = pg.normality(comparison[["cross_dataset_mae", "within_dataset_mae"]])
    print("Normality Test Results:\n", normality_results)

    wilcoxon_results = pg.wilcoxon(comparison["cross_dataset_mae"], comparison["within_dataset_mae"])
    hedges_g = pg.compute_effsize(
        comparison["cross_dataset_mae"], comparison["within_dataset_mae"], paired=True, eftype="hedges"
    )
    wilcoxon_results["hedges_g"] = hedges_g
    print("\nWilcoxon signed-rank test (subject-wise):\n", wilcoxon_results)
    return comparison, wilcoxon_results


comparison_guardian, wilcoxon_guardian = compare_cross_vs_within(
    results_empkins_to_guardian["subject_mae"], baseline_guardian_subject_mae, test_name="Guardian"
)

Test study: Guardian (24 matched participants)
Normality Test Results:
                            W      pval  normal
cross_dataset_mae   0.805152  0.000358   False
within_dataset_mae  0.929827  0.096624    True

Wilcoxon signed-rank test (subject-wise):
           W-val alternative     p-val   RBC      CLES  hedges_g
Wilcoxon   12.0   two-sided  0.000008  0.92  0.828125  1.120067


In [15]:
comparison_empkins, wilcoxon_empkins = compare_cross_vs_within(
    results_guardian_to_empkins["subject_mae"], baseline_empkins_subject_mae, test_name="EmpkinS"
)

Test study: EmpkinS (15 matched participants)
Normality Test Results:
                            W      pval  normal
cross_dataset_mae   0.885173  0.056711    True
within_dataset_mae  0.976732  0.942285    True

Wilcoxon signed-rank test (subject-wise):
           W-val alternative     p-val  RBC      CLES  hedges_g
Wilcoxon    0.0   two-sided  0.000061  1.0  0.968889  1.519572


## Why does cross-dataset performance degrade?

The target and features here are **absolute millisecond positions within the cardiac cycle** (e.g. the B-point sample position, RR-interval), not values normalized to a common physiological scale. EmpkinS (a psychosocial stress protocol) and Guardian (an autonomic-maneuver protocol) induce very different heart rates, so the two studies occupy different, only partially overlapping numeric ranges for both the inputs and the target.

`RandomForestRegressor` (like any tree ensemble) predicts via leaf averages and therefore **cannot extrapolate beyond the range of `y` values seen during training**. If the test study's targets lie partly outside the training study's target range, predictions for those samples are structurally capped at the training boundary. The cells below quantify this directly: the range/overlap of targets and features between the two studies, and — for each trained cross-dataset model — how much of the test set falls outside the training range, how clipped the resulting predictions are, and the resulting bias.

In [16]:
### Target and feature range overlap between studies
target_ranges = pd.DataFrame(
    {
        "EmpkinS": [
            data_empkins.index.get_level_values("b_point_sample_reference").min(),
            data_empkins.index.get_level_values("b_point_sample_reference").max(),
        ],
        "Guardian": [
            data_guardian.index.get_level_values("b_point_sample_reference").min(),
            data_guardian.index.get_level_values("b_point_sample_reference").max(),
        ],
    },
    index=["min", "max"],
)
print("Target (b_point_sample_reference) range per study [ms]:")
print(target_ranges)
print()

feature_ranges = pd.DataFrame(
    {
        "EmpkinS_min": data_empkins.min(),
        "EmpkinS_max": data_empkins.max(),
        "Guardian_min": data_guardian.min(),
        "Guardian_max": data_guardian.max(),
    }
)
print("Feature ranges per study [ms]:")
feature_ranges

Target (b_point_sample_reference) range per study [ms]:
     EmpkinS  Guardian
min    143.0     214.0
max    499.0     650.0

Feature ranges per study [ms]:


,EmpkinS_min,EmpkinS_max,Guardian_min,Guardian_max
rr_interval_ms_estimated,380.0,1272.0,434.0,1534.0
debski1993-second-derivative,137.0,528.0,232.0,708.0
drost2022,-87.0,545.0,-10.0,646.0
lozano2007-linear-regression,85.0,541.0,174.0,654.0


In [17]:
### Extrapolation / clipping diagnostic for each cross-dataset direction
def diagnose_extrapolation(results: dict[str, object]) -> None:
    """Report how far the test targets, features and predictions fall outside the training range."""
    y_train, y_test, y_pred = results["y_train"], results["y_test"], results["y_pred"]
    train_name, test_name = results["train_name"], results["test_name"]

    frac_target_out_of_range = np.mean((y_test < y_train.min()) | (y_test > y_train.max()))

    x_train, x_test = results["X_train"], results["X_test"]
    out_of_range_features = (x_test < np.nanmin(x_train, axis=0)) | (x_test > np.nanmax(x_train, axis=0))
    frac_features_out_of_range = np.mean(np.any(out_of_range_features, axis=1))

    bias = np.mean(y_pred - y_test)

    print(f"--- Train: {train_name} -> Test: {test_name} ---")
    print(f"Train target range: [{y_train.min():.0f}, {y_train.max():.0f}] ms")
    print(f"Test target range:  [{y_test.min():.0f}, {y_test.max():.0f}] ms")
    print(f"Fraction of {test_name} targets outside {train_name} training target range: {frac_target_out_of_range:.1%}")
    print(
        f"Fraction of {test_name} samples with >=1 feature outside {train_name} training feature range: "
        f"{frac_features_out_of_range:.1%}"
    )
    print(f"Model prediction range on {test_name}: [{y_pred.min():.0f}, {y_pred.max():.0f}] ms")
    print(f"Mean bias (prediction - true): {bias:+.2f} ms")
    print()


diagnose_extrapolation(results_empkins_to_guardian)
diagnose_extrapolation(results_guardian_to_empkins)

--- Train: EmpkinS -> Test: Guardian ---
Train target range: [143, 499] ms
Test target range:  [214, 650] ms
Fraction of Guardian targets outside EmpkinS training target range: 14.8%
Fraction of Guardian samples with >=1 feature outside EmpkinS training feature range: 11.9%
Model prediction range on Guardian: [233, 484] ms
Mean bias (prediction - true): -20.27 ms

--- Train: Guardian -> Test: EmpkinS ---
Train target range: [214, 650] ms
Test target range:  [143, 499] ms
Fraction of EmpkinS targets outside Guardian training target range: 19.2%
Fraction of EmpkinS samples with >=1 feature outside Guardian training feature range: 33.3%
Model prediction range on EmpkinS: [265, 517] ms
Mean bias (prediction - true): +51.99 ms



## Does the established best-performing pipeline generalize better?

The diagnostics above point to tree-based extrapolation as the root cause, which raises the question of whether a different model configuration generalizes better across studies. Rather than trying arbitrary alternative models, this uses the **full-feature, median-imputed dataset that is already precomputed** (`train_data_b_point_rr_interval_median_imputed.csv`, all 12 classical B-point algorithm outputs + RR-interval, NaNs already median-imputed) together with the pipeline that is established elsewhere in this project as the best-performing configuration *for that specific dataset*: `StandardScaler + SelectFromModel + RandomForestRegressor` (labeled `SS_SFM_RFR_RR-Interval-Median-Imputed` in `Compute_Statistical_Significance.ipynb`).

The exact per-fold tuned hyperparameters for this particular pipeline are not available locally (no cached `SklearnPipelinePermuter` pickle for the median-imputed data, unlike for the `include_nan` full-feature and `three_best_only` pipelines). The `RandomForestRegressor` step therefore reuses the project's documented best-performing hyperparameters (cf. `Computational_Cost.ipynb`), and `SelectFromModel` uses its default configuration (`RandomForestRegressor(n_estimators=100, random_state=0)` as the importance estimator, `threshold="mean"`), matching the defaults defined in `regression/hpc_scripts/b_point_regression_without_nan.py`.

In [18]:
# Load the full-feature, median-imputed dataset (already precomputed, no NaNs)
input_data_median_imputed = pd.read_csv(
    data_path.joinpath(f"rr-interval/{rater}/train_data_b_point_rr_interval_median_imputed.csv"),
    index_col=[0, 1, 2, 3, 4, 5],
).astype(float)

participant_ids_mi = input_data_median_imputed.index.get_level_values("participant")
data_empkins_mi = input_data_median_imputed.loc[participant_ids_mi.str.startswith("VP")]
data_guardian_mi = input_data_median_imputed.loc[participant_ids_mi.str.startswith("GDN")]

print(f"EmpkinS: {data_empkins_mi.shape[0]} heartbeats, {data_empkins_mi.shape[1]} features")
print(f"Guardian: {data_guardian_mi.shape[0]} heartbeats, {data_guardian_mi.shape[1]} features")
assert not data_empkins_mi.isna().any().any(), "Unexpected NaNs in median-imputed EmpkinS data"
assert not data_guardian_mi.isna().any().any(), "Unexpected NaNs in median-imputed Guardian data"

EmpkinS: 4935 heartbeats, 13 features
Guardian: 6203 heartbeats, 13 features


In [19]:
def make_pipeline_sfm_rfr_median_imputed() -> Pipeline:
    """Create the StandardScaler + SelectFromModel + RandomForestRegressor pipeline."""
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "reduce_dim",
                SelectFromModel(estimator=RandomForestRegressor(n_estimators=100, random_state=0), threshold="mean"),
            ),
            (
                "clf",
                RandomForestRegressor(
                    bootstrap=True,
                    criterion="friedman_mse",
                    max_features=0.8,
                    n_estimators=250,
                    ccp_alpha=0.001,
                    random_state=0,
                ),
            ),
        ]
    )


results_empkins_to_guardian_sfm = evaluate_cross_dataset(
    train_df=data_empkins_mi,
    test_df=data_guardian_mi,
    train_name="EmpkinS",
    test_name="Guardian",
    pipeline_factory=make_pipeline_sfm_rfr_median_imputed,
)
print()
results_guardian_to_empkins_sfm = evaluate_cross_dataset(
    train_df=data_guardian_mi,
    test_df=data_empkins_mi,
    train_name="Guardian",
    test_name="EmpkinS",
    pipeline_factory=make_pipeline_sfm_rfr_median_imputed,
)

Train: EmpkinS (n=4935) -> Test: Guardian (n=6203)
Sample-wise MAE: 32.60 ± 45.46 ms
Subject-wise MAE: 36.24 ± 25.74 ms

Train: Guardian (n=6203) -> Test: EmpkinS (n=4935)
Sample-wise MAE: 84.47 ± 87.99 ms
Subject-wise MAE: 72.66 ± 63.80 ms


In [20]:
diagnose_extrapolation(results_empkins_to_guardian_sfm)
diagnose_extrapolation(results_guardian_to_empkins_sfm)

--- Train: EmpkinS -> Test: Guardian ---
Train target range: [143, 499] ms
Test target range:  [214, 650] ms
Fraction of Guardian targets outside EmpkinS training target range: 14.8%
Fraction of Guardian samples with >=1 feature outside EmpkinS training feature range: 16.8%
Model prediction range on Guardian: [180, 481] ms
Mean bias (prediction - true): -24.57 ms

--- Train: Guardian -> Test: EmpkinS ---
Train target range: [214, 650] ms
Test target range:  [143, 499] ms
Fraction of EmpkinS targets outside Guardian training target range: 19.3%
Fraction of EmpkinS samples with >=1 feature outside Guardian training feature range: 8.8%
Model prediction range on EmpkinS: [240, 537] ms
Mean bias (prediction - true): +78.43 ms



In [21]:
comparison_models_df = pd.DataFrame(
    [
        {
            "setting": "EmpkinS -> Guardian",
            "model": "MinMax_RFR (three_best_only)",
            "sample_wise_mae": fmt_mean_std(
                results_empkins_to_guardian["abs_error"].mean(), results_empkins_to_guardian["abs_error"].std()
            ),
            "subject_wise_mae": fmt_mean_std(
                results_empkins_to_guardian["subject_mae"].mean(), results_empkins_to_guardian["subject_mae"].std()
            ),
        },
        {
            "setting": "EmpkinS -> Guardian",
            "model": "SS_SFM_RFR (median_imputed, established best)",
            "sample_wise_mae": fmt_mean_std(
                results_empkins_to_guardian_sfm["abs_error"].mean(), results_empkins_to_guardian_sfm["abs_error"].std()
            ),
            "subject_wise_mae": fmt_mean_std(
                results_empkins_to_guardian_sfm["subject_mae"].mean(),
                results_empkins_to_guardian_sfm["subject_mae"].std(),
            ),
        },
        {
            "setting": "Guardian -> EmpkinS",
            "model": "MinMax_RFR (three_best_only)",
            "sample_wise_mae": fmt_mean_std(
                results_guardian_to_empkins["abs_error"].mean(), results_guardian_to_empkins["abs_error"].std()
            ),
            "subject_wise_mae": fmt_mean_std(
                results_guardian_to_empkins["subject_mae"].mean(), results_guardian_to_empkins["subject_mae"].std()
            ),
        },
        {
            "setting": "Guardian -> EmpkinS",
            "model": "SS_SFM_RFR (median_imputed, established best)",
            "sample_wise_mae": fmt_mean_std(
                results_guardian_to_empkins_sfm["abs_error"].mean(), results_guardian_to_empkins_sfm["abs_error"].std()
            ),
            "subject_wise_mae": fmt_mean_std(
                results_guardian_to_empkins_sfm["subject_mae"].mean(),
                results_guardian_to_empkins_sfm["subject_mae"].std(),
            ),
        },
    ]
)
comparison_models_df

,setting,model,sample_wise_mae,subject_wise_mae
0,EmpkinS -> Guardian,MinMax_RFR (three_best_only),26.67 ± 34.71,29.77 ± 22.25
1,EmpkinS -> Guardian,"SS_SFM_RFR (median_imputed, established best)",32.60 ± 45.46,36.24 ± 25.74
2,Guardian -> EmpkinS,MinMax_RFR (three_best_only),60.13 ± 56.86,52.92 ± 39.35
3,Guardian -> EmpkinS,"SS_SFM_RFR (median_imputed, established best)",84.47 ± 88.00,72.66 ± 63.80


**Result:** the established best-performing pipeline for the median-imputed data does not fix cross-dataset generalization — it is still a `RandomForestRegressor` at its core, so it inherits the same inability to extrapolate beyond the training target/feature range. `SelectFromModel` additionally collapses the feature set down to a single algorithm estimate (`drost2022`) in both directions, discarding `rr_interval_ms_estimated` — the feature most directly tied to the heart-rate shift between studies — which removes information the model could otherwise have used to adapt, rather than adding robustness. The takeaway is that better within-dataset performance does not imply better cross-dataset generalization: the fix has to address the tree-based extrapolation limitation itself (e.g. a linear/extrapolation-capable model, or features normalized relative to the cardiac cycle instead of absolute milliseconds), not just pick a "better" model by within-dataset metrics.

## Next best *non*-tree-based method

Since the established best pipeline is still tree-based and inherits the extrapolation problem, this checks the next best-performing **non**-tree-based model configuration among those already evaluated for the median-imputed dataset (`train_data_b_point_rr_interval_median_imputed_ml_results.csv`), ranked by within-dataset MAE.

In [22]:
# Rank all evaluated model configurations for the median-imputed dataset by within-dataset MAE
ml_results_median_imputed = pd.read_csv(
    data_path.joinpath(f"rr-interval/{rater}/train_data_b_point_rr_interval_median_imputed_ml_results.csv")
)
model_cols = [c for c in ml_results_median_imputed.columns if "RR-Interval-Median-Imputed" in c]
y_ref = ml_results_median_imputed["b_point_sample_reference"]

mae_ranking = (
    pd.Series({c: (ml_results_median_imputed[c] - y_ref).abs().mean() for c in model_cols})
    .sort_values()
    .rename("within_dataset_mae")
)
tree_based = mae_ranking.index.str.contains("RFR|DTR")
mae_ranking.to_frame().assign(tree_based=tree_based)

,within_dataset_mae,tree_based
SS_SFM_RFR_RR-Interval-Median-Imputed,8.303370,True
MinMax_KBest_RFR_RR-Interval-Median-Imputed,8.332127,True
MinMax_SFM_RFR_RR-Interval-Median-Imputed,8.404488,True
SS_KBest_RFR_RR-Interval-Median-Imputed,8.417848,True
SS_KBest_SVR_RR-Interval-Median-Imputed,8.471520,False
MinMax_SFM_DTR_RR-Interval-Median-Imputed,8.625426,True
MinMax_KBest_DTR_RR-Interval-Median-Imputed,8.790088,True
SS_SFM_DTR_RR-Interval-Median-Imputed,8.792737,True
SS_SFM_SVR_RR-Interval-Median-Imputed,8.801365,False
SS_KBest_DTR_RR-Interval-Median-Imputed,8.998698,True


The best non-tree-based configuration is `SS_KBest_SVR_RR-Interval-Median-Imputed` (`StandardScaler + SelectKBest + SVR`). As with `SS_SFM_RFR`, no cached hyperparameter-search result is available locally for this exact pipeline, so its hyperparameters (`SelectKBest`'s `k`, `SVR`'s `kernel`/`C`/`epsilon`/`gamma`) are tuned here with `GridSearchCV` over the same search space used in `regression/hpc_scripts/b_point_regression_without_nan.py`, using 5-fold group CV **within the training study only** (no leakage from the held-out study) — the same nested-CV principle used to originally select this pipeline.

In [23]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

svr_param_grid = [
    {"kbest__k": [4, 8, "all"], "svr__kernel": ["linear"], "svr__C": [0.1, 1, 10, 100], "svr__epsilon": [0.01, 0.1, 1]},
    {
        "kbest__k": [4, 8, "all"],
        "svr__kernel": ["rbf"],
        "svr__C": [0.1, 1, 10, 100],
        "svr__gamma": [0.001, 0.01, 0.1, 1],
        "svr__epsilon": [0.01, 0.1, 1],
    },
]


def evaluate_cross_dataset_tuned_svr(
    train_df: pd.DataFrame, test_df: pd.DataFrame, train_name: str, test_name: str
) -> dict[str, object]:
    """Tune an SVR on one dataset with grouped CV and evaluate it on the other dataset."""
    x_train, y_train, groups_train, _ = bp.classification.utils.prepare_df_sklearn(
        data=train_df, label_col="b_point_sample_reference", subject_col="participant", print_summary=False
    )
    x_test, y_test, _, _ = bp.classification.utils.prepare_df_sklearn(
        data=test_df, label_col="b_point_sample_reference", subject_col="participant", print_summary=False
    )

    base_pipeline = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("kbest", SelectKBest(score_func=f_regression)),
            ("svr", SVR()),
        ]
    )
    search = GridSearchCV(
        base_pipeline, svr_param_grid, cv=GroupKFold(n_splits=5), scoring="neg_mean_absolute_error", n_jobs=-1
    )
    search.fit(x_train, y_train, groups=groups_train)
    print(f"Train: {train_name} -> Test: {test_name}")
    print(f"Selected hyperparameters: {search.best_params_}")
    print(f"Inner-CV (within {train_name}) MAE: {-search.best_score_:.2f} ms")

    pipeline = search.best_estimator_
    y_pred = pipeline.predict(x_test)

    abs_error = np.abs(y_test - y_pred)
    mae_sample_wise = np.mean(abs_error)
    std_sample_wise = np.std(abs_error)
    print(f"Sample-wise MAE: {mae_sample_wise:.2f} ± {std_sample_wise:.2f} ms")

    abs_error_series = pd.Series(abs_error, index=test_df.index, name="abs_error")
    subject_mae = abs_error_series.groupby("participant").mean()
    print(f"Subject-wise MAE: {subject_mae.mean():.2f} ± {subject_mae.std():.2f} ms")

    return {
        "train_name": train_name,
        "test_name": test_name,
        "pipeline": pipeline,
        "X_train": x_train,
        "y_train": y_train,
        "X_test": x_test,
        "y_test": y_test,
        "y_pred": y_pred,
        "abs_error": abs_error_series,
        "subject_mae": subject_mae,
        "mae_sample_wise": mae_sample_wise,
        "std_sample_wise": std_sample_wise,
    }


results_empkins_to_guardian_svr = evaluate_cross_dataset_tuned_svr(
    train_df=data_empkins_mi, test_df=data_guardian_mi, train_name="EmpkinS", test_name="Guardian"
)
print()
results_guardian_to_empkins_svr = evaluate_cross_dataset_tuned_svr(
    train_df=data_guardian_mi, test_df=data_empkins_mi, train_name="Guardian", test_name="EmpkinS"
)

Train: EmpkinS -> Test: Guardian
Selected hyperparameters: {'kbest__k': 8, 'svr__C': 100, 'svr__epsilon': 1, 'svr__gamma': 0.1, 'svr__kernel': 'rbf'}
Inner-CV (within EmpkinS) MAE: 8.76 ms
Sample-wise MAE: 25.65 ± 41.08 ms
Subject-wise MAE: 29.19 ± 27.41 ms

Train: Guardian -> Test: EmpkinS
Selected hyperparameters: {'kbest__k': 8, 'svr__C': 100, 'svr__epsilon': 1, 'svr__gamma': 0.1, 'svr__kernel': 'rbf'}
Inner-CV (within Guardian) MAE: 8.53 ms
Sample-wise MAE: 67.96 ± 75.71 ms
Subject-wise MAE: 58.02 ± 54.98 ms


In [24]:
diagnose_extrapolation(results_empkins_to_guardian_svr)
diagnose_extrapolation(results_guardian_to_empkins_svr)

--- Train: EmpkinS -> Test: Guardian ---
Train target range: [143, 499] ms
Test target range:  [214, 650] ms
Fraction of Guardian targets outside EmpkinS training target range: 14.8%
Fraction of Guardian samples with >=1 feature outside EmpkinS training feature range: 16.8%
Model prediction range on Guardian: [239, 482] ms
Mean bias (prediction - true): -16.22 ms

--- Train: Guardian -> Test: EmpkinS ---
Train target range: [214, 650] ms
Test target range:  [143, 499] ms
Fraction of EmpkinS targets outside Guardian training target range: 19.3%
Fraction of EmpkinS samples with >=1 feature outside Guardian training feature range: 8.8%
Model prediction range on EmpkinS: [263, 509] ms
Mean bias (prediction - true): +61.49 ms



In [25]:
comparison_models_df = pd.concat(
    [
        comparison_models_df,
        pd.DataFrame(
            [
                {
                    "setting": "EmpkinS -> Guardian",
                    "model": "SS_KBest_SVR (median_imputed, next best non-tree)",
                    "sample_wise_mae": fmt_mean_std(
                        results_empkins_to_guardian_svr["abs_error"].mean(),
                        results_empkins_to_guardian_svr["abs_error"].std(),
                    ),
                    "subject_wise_mae": fmt_mean_std(
                        results_empkins_to_guardian_svr["subject_mae"].mean(),
                        results_empkins_to_guardian_svr["subject_mae"].std(),
                    ),
                },
                {
                    "setting": "Guardian -> EmpkinS",
                    "model": "SS_KBest_SVR (median_imputed, next best non-tree)",
                    "sample_wise_mae": fmt_mean_std(
                        results_guardian_to_empkins_svr["abs_error"].mean(),
                        results_guardian_to_empkins_svr["abs_error"].std(),
                    ),
                    "subject_wise_mae": fmt_mean_std(
                        results_guardian_to_empkins_svr["subject_mae"].mean(),
                        results_guardian_to_empkins_svr["subject_mae"].std(),
                    ),
                },
            ]
        ),
    ],
    ignore_index=True,
)
comparison_models_df

,setting,model,sample_wise_mae,subject_wise_mae
0,EmpkinS -> Guardian,MinMax_RFR (three_best_only),26.67 ± 34.71,29.77 ± 22.25
1,EmpkinS -> Guardian,"SS_SFM_RFR (median_imputed, established best)",32.60 ± 45.46,36.24 ± 25.74
2,Guardian -> EmpkinS,MinMax_RFR (three_best_only),60.13 ± 56.86,52.92 ± 39.35
3,Guardian -> EmpkinS,"SS_SFM_RFR (median_imputed, established best)",84.47 ± 88.00,72.66 ± 63.80
4,EmpkinS -> Guardian,"SS_KBest_SVR (median_imputed, next best non-tree)",25.65 ± 41.08,29.19 ± 27.41
5,Guardian -> EmpkinS,"SS_KBest_SVR (median_imputed, next best non-tree)",67.96 ± 75.72,58.02 ± 54.98


**Result:** `GridSearchCV` selects an **RBF** kernel in both directions, because RBF fits the within-study training data better than a linear kernel (lower inner-CV MAE). But RBF, like `RandomForestRegressor`, is a local/non-parametric method that cannot extrapolate — it does not resolve the cross-dataset gap: EmpkinS→Guardian MAE (25.65 ms) is essentially unchanged from the tree-based model, and Guardian→EmpkinS MAE (67.96 ms) is *worse* than the tree-based model (60.13 ms).

This shows that "next best non-tree method" is not automatically "next best at generalizing" — an unconstrained hyperparameter search picks the kernel that fits the training distribution best, which is precisely the wrong criterion when the training and test studies have different distributions. The cell below checks what happens if the kernel is constrained to `linear` (still an SVR, still non-tree, just restricted to the one sub-family capable of extrapolating past the training range).

In [26]:
linear_svr_param_grid = {"kbest__k": [4, 8, "all"], "svr__C": [0.1, 1, 10, 100], "svr__epsilon": [0.01, 0.1, 1]}


def evaluate_cross_dataset_linear_svr(
    train_df: pd.DataFrame, test_df: pd.DataFrame, train_name: str, test_name: str
) -> dict[str, object]:
    """Tune a linear SVR on one dataset with grouped CV and evaluate it on the other dataset."""
    x_train, y_train, groups_train, _ = bp.classification.utils.prepare_df_sklearn(
        data=train_df, label_col="b_point_sample_reference", subject_col="participant", print_summary=False
    )
    x_test, y_test, _, _ = bp.classification.utils.prepare_df_sklearn(
        data=test_df, label_col="b_point_sample_reference", subject_col="participant", print_summary=False
    )

    base_pipeline = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("kbest", SelectKBest(score_func=f_regression)),
            ("svr", SVR(kernel="linear")),
        ]
    )
    search = GridSearchCV(
        base_pipeline, linear_svr_param_grid, cv=GroupKFold(n_splits=5), scoring="neg_mean_absolute_error", n_jobs=-1
    )
    search.fit(x_train, y_train, groups=groups_train)
    print(f"Train: {train_name} -> Test: {test_name}")
    print(f"Selected hyperparameters: {search.best_params_}")
    print(f"Inner-CV (within {train_name}) MAE: {-search.best_score_:.2f} ms")

    y_pred = search.best_estimator_.predict(x_test)
    abs_error = np.abs(y_test - y_pred)
    abs_error_series = pd.Series(abs_error, index=test_df.index, name="abs_error")
    subject_mae = abs_error_series.groupby("participant").mean()
    print(f"Sample-wise MAE: {abs_error.mean():.2f} ± {abs_error.std():.2f} ms")
    print(f"Subject-wise MAE: {subject_mae.mean():.2f} ± {subject_mae.std():.2f} ms")
    print(f"Mean bias (prediction - true): {np.mean(y_pred - y_test):+.2f} ms")

    return {
        "train_name": train_name,
        "test_name": test_name,
        "pipeline": search.best_estimator_,
        "abs_error": abs_error_series,
        "subject_mae": subject_mae,
    }


results_empkins_to_guardian_linsvr = evaluate_cross_dataset_linear_svr(
    train_df=data_empkins_mi, test_df=data_guardian_mi, train_name="EmpkinS", test_name="Guardian"
)
print()
results_guardian_to_empkins_linsvr = evaluate_cross_dataset_linear_svr(
    train_df=data_guardian_mi, test_df=data_empkins_mi, train_name="Guardian", test_name="EmpkinS"
)

Train: EmpkinS -> Test: Guardian
Selected hyperparameters: {'kbest__k': 'all', 'svr__C': 10, 'svr__epsilon': 0.01}
Inner-CV (within EmpkinS) MAE: 10.86 ms
Sample-wise MAE: 19.39 ± 17.06 ms
Subject-wise MAE: 19.74 ± 4.11 ms
Mean bias (prediction - true): +15.70 ms

Train: Guardian -> Test: EmpkinS
Selected hyperparameters: {'kbest__k': 'all', 'svr__C': 10, 'svr__epsilon': 1}
Inner-CV (within Guardian) MAE: 10.33 ms
Sample-wise MAE: 12.73 ± 25.18 ms
Subject-wise MAE: 12.70 ± 6.33 ms
Mean bias (prediction - true): -7.21 ms


In [27]:
comparison_models_df = pd.concat(
    [
        comparison_models_df,
        pd.DataFrame(
            [
                {
                    "setting": "EmpkinS -> Guardian",
                    "model": "SS_KBest_SVR(linear) (median_imputed, kernel constrained)",
                    "sample_wise_mae": fmt_mean_std(
                        results_empkins_to_guardian_linsvr["abs_error"].mean(),
                        results_empkins_to_guardian_linsvr["abs_error"].std(),
                    ),
                    "subject_wise_mae": fmt_mean_std(
                        results_empkins_to_guardian_linsvr["subject_mae"].mean(),
                        results_empkins_to_guardian_linsvr["subject_mae"].std(),
                    ),
                },
                {
                    "setting": "Guardian -> EmpkinS",
                    "model": "SS_KBest_SVR(linear) (median_imputed, kernel constrained)",
                    "sample_wise_mae": fmt_mean_std(
                        results_guardian_to_empkins_linsvr["abs_error"].mean(),
                        results_guardian_to_empkins_linsvr["abs_error"].std(),
                    ),
                    "subject_wise_mae": fmt_mean_std(
                        results_guardian_to_empkins_linsvr["subject_mae"].mean(),
                        results_guardian_to_empkins_linsvr["subject_mae"].std(),
                    ),
                },
            ]
        ),
    ],
    ignore_index=True,
)
comparison_models_df = comparison_models_df.sort_values("setting").reset_index(drop=True)
comparison_models_df

,setting,model,sample_wise_mae,subject_wise_mae
0,EmpkinS -> Guardian,MinMax_RFR (three_best_only),26.67 ± 34.71,29.77 ± 22.25
1,EmpkinS -> Guardian,"SS_SFM_RFR (median_imputed, established best)",32.60 ± 45.46,36.24 ± 25.74
2,EmpkinS -> Guardian,"SS_KBest_SVR (median_imputed, next best non-tree)",25.65 ± 41.08,29.19 ± 27.41
3,EmpkinS -> Guardian,"SS_KBest_SVR(linear) (median_imputed, kernel c...",19.39 ± 17.06,19.74 ± 4.11
4,Guardian -> EmpkinS,MinMax_RFR (three_best_only),60.13 ± 56.86,52.92 ± 39.35
5,Guardian -> EmpkinS,"SS_SFM_RFR (median_imputed, established best)",84.47 ± 88.00,72.66 ± 63.80
6,Guardian -> EmpkinS,"SS_KBest_SVR (median_imputed, next best non-tree)",67.96 ± 75.72,58.02 ± 54.98
7,Guardian -> EmpkinS,"SS_KBest_SVR(linear) (median_imputed, kernel c...",12.73 ± 25.19,12.70 ± 6.33


In [28]:
print(comparison_models_df.to_latex(index=False).replace("±", r"$\pm$"))

\begin{tabular}{llll}
\toprule
setting & model & sample_wise_mae & subject_wise_mae \\
\midrule
EmpkinS -> Guardian & MinMax_RFR (three_best_only) & 26.67 $\pm$ 34.71 & 29.77 $\pm$ 22.25 \\
EmpkinS -> Guardian & SS_SFM_RFR (median_imputed, established best) & 32.60 $\pm$ 45.46 & 36.24 $\pm$ 25.74 \\
EmpkinS -> Guardian & SS_KBest_SVR (median_imputed, next best non-tree) & 25.65 $\pm$ 41.08 & 29.19 $\pm$ 27.41 \\
EmpkinS -> Guardian & SS_KBest_SVR(linear) (median_imputed, kernel constrained) & 19.39 $\pm$ 17.06 & 19.74 $\pm$ 4.11 \\
Guardian -> EmpkinS & MinMax_RFR (three_best_only) & 60.13 $\pm$ 56.86 & 52.92 $\pm$ 39.35 \\
Guardian -> EmpkinS & SS_SFM_RFR (median_imputed, established best) & 84.47 $\pm$ 88.00 & 72.66 $\pm$ 63.80 \\
Guardian -> EmpkinS & SS_KBest_SVR (median_imputed, next best non-tree) & 67.96 $\pm$ 75.72 & 58.02 $\pm$ 54.98 \\
Guardian -> EmpkinS & SS_KBest_SVR(linear) (median_imputed, kernel constrained) & 12.73 $\pm$ 25.19 & 12.70 $\pm$ 6.33 \\
\bottomrule
\end{tab

**Summary:** among the model families actually evaluated in this project, the picture is:

- The overall best-performing pipeline (`SS_SFM_RFR`, full features, median-imputed) and the current `three_best_only` pipeline are both tree-based and both suffer badly from cross-dataset extrapolation.
- The next best *non*-tree-based pipeline as selected by an unconstrained hyperparameter search (`SS_KBest_SVR`, RBF kernel) does **not** fix this — it picks RBF because RBF fits the training study best, and RBF has the same locality/extrapolation limitation as trees.
- Restricting the SVR to a **linear** kernel — a strictly worse choice by within-dataset MAE, which is why the unconstrained search never selects it — generalizes substantially better cross-dataset in both directions, because a linear model's predictions are not bounded by the training target range.

So the answer to "is this better with a different model" depends on what "better" is optimized for: picking a model by within-dataset MAE (which is what all of these established "best" pipelines were selected by) does not select for cross-dataset robustness, and can even select against it.